# Kernel selection using MAP and likelihood

In [ ]:
import deepinv as dinv
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
import numpy as np
from priors import L2, GSDPrior
import tqdm as tqdm
from experiments_utils import generate_blur_operator
from deepinv.physics.blur import gaussian_blur

from utils import moffat, laplace, uniform
device = torch.device("cuda")

sigma = 0.1

denoiser = dinv.models.GSDRUNet(pretrained="models/GSDRUNet_grayscale_torch.ckpt", 
                                in_channels=1, out_channels=1, device=device)

kernels = [gaussian_blur(sigma=(2, 2)),
           moffat((0.5, 1), size=7),
           laplace(0.4, size=10), uniform(3), 
           gaussian_blur(sigma=(2.5, 2.5)),]

# determining max padding to remove from the images (due to FFT circular padding in deepinv)
cuts = np.zeros(5, dtype=int)
xc = torch.zeros(1, 1, 256, 256).to(device)
for i in range(5):
    cuts[i] = (256 -  dinv.physics.Blur(img_size=(1, 256, 256), filter=kernels[i], 
                       device=device, padding='valid')(xc).shape[-1])//2
max_cut = np.max(cuts)
del xc

sigma_denoiser = 0.1

data_fidelity = dinv.optim.data_fidelity.L2(sigma)

In [ ]:
def GradientStepPnP(z0, y, f, g, physics, sigma_denoiser, tau0, eta=0.9, gamma=0.1, 
                    max_iter=700, thresh_conv=1e-6):
    """ Implementation of Gradient Step Plug and Play from "Gradient Step Denoiser for 
    convergent Plug-and-Play (ICLR 2022)"""
    
    with torch.no_grad():
        lam = g.param.clone()
        tau = tau0/eta

        xk = f.prox(z0, y, physics, gamma=tau)
        k, delta = 0, thresh_conv + 1
        fun0 = g(xk) + f(xk, y, physics)
        fvals = [fun0.cpu().item()]
        taus = [tau]
        while k < max_iter and delta > thresh_conv:
            zk = lam*tau*g.denoiser(xk, sigma_denoiser) + (1-lam*tau)*xk
            xkp = f.prox(zk, y, physics, gamma=tau)
            fval = g(xkp) + f(xkp, y, physics)
            diff = fvals[-1] - fval
            if diff < gamma*torch.norm(xk-xkp)**2/tau:
                tau = eta*tau
            else: 
                delta = diff / fun0
                xk = xkp.clone()
                fvals.append(fval.cpu().item())
                taus.append(tau)

                k += 1
        xhat = lam*tau*g.denoiser(xk, sigma_denoiser) + (1-lam*tau)*xk
    return xhat, k, delta, fvals, taus

In [ ]:
tune_lam = False
psnrs = np.zeros([5, 5, 3])
likelihoods = np.zeros([5, 5, 3])
xhats = np.zeros([5, 5, 3, 1, 1, 256, 256])
psnr = dinv.loss.metric.PSNR()
for ind_gt in range(5):
    for ind_kernel in tqdm.tqdm(range(5)):
        if tune_lam:
            best_lam = np.load("results/kernel_comparison/sapg_res_{}_{}.npy".format(ind_gt, ind_kernel))
        for im_ind in range(3):
            if tune_lam:
                lam = best_lam[im_ind]
            else:
                lam = 110.
            with torch.no_grad():
                
                trace = np.load("results/kernel_comparison/trace_{}_{}_{}.npz".format(ind_gt, 0, im_ind))
                x, y = torch.tensor(trace["x"], device=device), torch.tensor(trace["y"], device=device)
                physics =  generate_blur_operator(256, filter_torch=kernels[ind_kernel], sigma=0)
    
                g = GSDPrior(torch.tensor(lam, device=device), denoiser, sigma_denoiser=0.1)
                xhat, k, delta, fvals, taus = GradientStepPnP(y, y, data_fidelity, g, physics, 0.1, 
                                                              50/lam, max_iter=500, thresh_conv=1e-4, eta=0.9, gamma=0.1, )
                
                xhats[ind_gt, ind_kernel, im_ind] = xhat.cpu().numpy()
                xhat_cut = xhat[:,:,max_cut:256-max_cut,max_cut:256-max_cut]
                x_cut = x[:,:,max_cut:256-max_cut,max_cut:256-max_cut]
                x_proj = physics(xhat)
                x_proj = x_proj[:,:,max_cut:256-max_cut,max_cut:256-max_cut]
                y_cut = y[:,:,max_cut:256-max_cut,max_cut:256-max_cut]
                psnrs[ind_gt, ind_kernel, im_ind] = psnr(x_cut, xhat_cut).cpu().numpy()
                likelihoods[ind_gt, ind_kernel, im_ind] = np.linalg.norm((x_proj - y_cut).cpu().numpy())
if tune_lam:
    np.savez("results/kernel_comparison/res_map_new.npz", likelihoods=likelihoods, psnrs=psnrs, xhats=xhats)
else:
    np.savez("results/kernel_comparison/res_map_no_tune_new.npz", likelihoods=likelihoods, psnrs=psnrs, xhats=xhats)

In [ ]:
# compute single and few shots accuracy (likelihoods is negative log likelihood here)
res_tune = np.load("results/kernel_comparison/res_map_new.npz")
res_notune = np.load("results/kernel_comparison/res_map_no_tune_new.npz")
acc_tune, acc_notune = 0, 0
amins_tune = np.argmin(res_tune["likelihoods"], axis=1)
amins_notune = np.argmin(res_notune["likelihoods"], axis=1)
amins_mean_tune = np.argmin(np.mean(res_tune["likelihoods"]**2, axis=2), axis=1)
amins_mean_notune = np.argmin(np.mean(res_notune["likelihoods"]**2, axis=2), axis=1)
for i in range(5):
    for j in range(3):
        if amins_tune[i, j] == i:
            acc_tune += 1
        if amins_notune[i, j] == i:
            acc_notune += 1
print(acc_notune / 15 *100, "&", np.mean(amins_mean_notune==np.arange(5))*100)

print(acc_tune / 15*100, "&", np.mean(amins_mean_tune==np.arange(5))*100)


In [ ]:
kernel_names = [r"$\kappa_{\mathcal G}(2)$", r"$\kappa_{\mathcal M}(0.5, 1)$", r"$\kappa_{\mathcal L}(0.4)$", r"$\kappa_{\mathcal U}(3)$", r"$\kappa_{\mathcal G}(2.5)$"]

mean_full = res_tune["likelihoods"]**2 - 550
mean_on_im = np.mean(mean_full, axis=2)
min_ind = np.argmin(mean_on_im, axis=1)
for i in range(len(mean_on_im)):
    line_mean = kernel_names[i] + " & "
    for j in range(mean_on_im.shape[1]):  # main column
        if min_ind[i] == j:
            line_mean += "\\textbf{{{:.3f}}} & ".format(mean_on_im[i, j])
        else:
            line_mean += "{:.3f} & ".format(mean_on_im[i, j])
    print(line_mean[:-2] + "\\\\")   